# FreshGuard CNN Fine-tuning Framework
Modular pipeline — define `ModelConfig` objects and call `compare_models` to train and compare any supported CNN backbone.
Supports **Google Colab**, **checkpoint resume**, and **per-batch progress bars**.

## Environment Setup

**Run exactly one of the two cells below** depending on where you are running:
- **Colab cell**: mounts Google Drive, sets Drive paths, creates checkpoint dirs
- **Local cell**: sets local `./` paths — skip if you are on Colab

In [ ]:
# =============================================================================
# GOOGLE COLAB SETUP
# Run this cell ONLY when using Google Colab. Skip if running locally.
# =============================================================================

# from google.colab import drive
# drive.mount('/content/drive')

# import os

# # Primary checkpoint dir — local Colab SSD (fast writes during training)
# CHECKPOINT_DIR        = "/content/checkpoints"

# # Backup checkpoint dir — Google Drive (survives runtime disconnects)
# BACKUP_CHECKPOINT_DIR = "/content/drive/MyDrive/FreshGuard/checkpoints"

# DATA_TRAIN_DIR = "/content/drive/MyDrive/FreshGuard/data/train"
# DATA_VAL_DIR   = "/content/drive/MyDrive/FreshGuard/data/val"
# MODEL_SAVE_DIR = "/content/drive/MyDrive/FreshGuard/models"

# for d in [CHECKPOINT_DIR, BACKUP_CHECKPOINT_DIR, MODEL_SAVE_DIR]:
#     os.makedirs(d, exist_ok=True)

# print("Colab environment ready.")
# print(f"  Checkpoint (local)  : {CHECKPOINT_DIR}")
# print(f"  Checkpoint (Drive)  : {BACKUP_CHECKPOINT_DIR}")
# print(f"  Train data          : {DATA_TRAIN_DIR}")
# print(f"  Val data            : {DATA_VAL_DIR}")
# print(f"  Model save dir      : {MODEL_SAVE_DIR}")

In [ ]:
# =============================================================================
# LOCAL SETUP
# Run this cell ONLY when running locally. Skip if using Google Colab.
# =============================================================================

import os

CHECKPOINT_DIR        = "./checkpoints"
BACKUP_CHECKPOINT_DIR = None            # no Drive backup needed locally
DATA_TRAIN_DIR        = "./data/train"
DATA_VAL_DIR          = "./data/val"
MODEL_SAVE_DIR        = "./models"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR,  exist_ok=True)

print("Local environment ready.")
print(f"  Checkpoint dir : {CHECKPOINT_DIR}")
print(f"  Train data     : {DATA_TRAIN_DIR}")
print(f"  Val data       : {DATA_VAL_DIR}")
print(f"  Model save dir : {MODEL_SAVE_DIR}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from pathlib import Path
import random
import copy
import time
import json
from dataclasses import dataclass
from typing import List, Optional, Dict

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm not found — falling back to plain print progress")

In [ ]:
def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using GPU:", torch.cuda.get_device_name(0))
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU")
    else:
        device = torch.device("cpu")
        print("Using CPU")
    return device

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = get_device()
set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## ModelConfig — per-model hyperparameters

Each `ModelConfig` carries everything needed to build, normalise, and train one backbone:
- `name`: key into the registry (`"resnet50"`, `"resnet18"`, `"mobilenet_v2"`, `"efficientnet_b0"`)
- `mean` / `std`: per-channel normalisation stats
- `resize_size`: val `Resize()` target — ResNet-50 V2 weights use 232, all others use 256
- `lr`, `weight_decay`, `epochs`: optimiser / scheduler knobs

In [ ]:
@dataclass
class ModelConfig:
    name: str
    mean: List[float]
    std:  List[float]
    lr:   float
    epochs: int
    weight_decay: float = 1e-4
    batch_size: int = 64          # per-model batch size for VRAM tuning
    resize_size: int = 256        # val Resize() target; V2 weights use 232
    t_max: Optional[int] = None  # CosineAnnealingLR T_max; defaults to epochs

    def __post_init__(self):
        if self.t_max is None:
            self.t_max = self.epochs


# get_head returns the final Linear; set_head replaces it.
_MODEL_REGISTRY: Dict[str, Dict] = {
    "resnet50": {
        "fn":       models.resnet50,
        "weights":  models.ResNet50_Weights.IMAGENET1K_V2,
        "get_head": lambda m: m.fc,
        "set_head": lambda m, h: setattr(m, "fc", h),
    },
    "resnet18": {
        "fn":       models.resnet18,
        "weights":  models.ResNet18_Weights.IMAGENET1K_V1,
        "get_head": lambda m: m.fc,
        "set_head": lambda m, h: setattr(m, "fc", h),
    },
    "mobilenet_v2": {
        "fn":       models.mobilenet_v2,
        "weights":  models.MobileNet_V2_Weights.IMAGENET1K_V1,
        "get_head": lambda m: m.classifier[1],
        "set_head": lambda m, h: m.classifier.__setitem__(1, h),
    },
    "efficientnet_b0": {
        "fn":       models.efficientnet_b0,
        "weights":  models.EfficientNet_B0_Weights.IMAGENET1K_V1,
        "get_head": lambda m: m.classifier[1],
        "set_head": lambda m, h: m.classifier.__setitem__(1, h),
    },
}


def build_model(config: ModelConfig, num_classes: int) -> torch.nn.Module:
    if config.name not in _MODEL_REGISTRY:
        raise ValueError(
            f"Unknown model '{config.name}'. Available: {list(_MODEL_REGISTRY)}"
        )
    entry = _MODEL_REGISTRY[config.name]
    model = entry["fn"](weights=entry["weights"])
    for param in model.parameters():
        param.requires_grad = True
    in_features = entry["get_head"](model).in_features
    entry["set_head"](model, torch.nn.Linear(in_features, num_classes))
    return model

## Data Pipeline

`get_transforms` builds train/val pipelines from the config's normalisation stats.  
`get_dataloaders` wraps those into `ImageFolder` + `DataLoader` objects.  
`num_workers=2` is used — reliable on both Colab and local.

In [ ]:
def get_transforms(mean: List[float], std: List[float], resize_size: int = 256):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(resize_size),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tf, val_tf


def get_dataloaders(train_dir: str, val_dir: str,
                    mean: List[float], std: List[float],
                    resize_size: int = 256,
                    batch_size: int = 64,
                    device: Optional[torch.device] = None):
    train_tf, val_tf = get_transforms(mean, std, resize_size)
    train_ds = datasets.ImageFolder(root=train_dir, transform=train_tf)
    val_ds   = datasets.ImageFolder(root=val_dir,   transform=val_tf)

    assert train_ds.classes == val_ds.classes, (
        "Class mismatch between train and val folders. "
        f"Train: {train_ds.classes} | Val: {val_ds.classes}"
    )

    pin        = device is not None and device.type == "cuda"
    n_workers  = 2  # 2 is stable on both Colab and local; 4 can cause Colab hangs
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=n_workers, pin_memory=pin,
                              persistent_workers=(n_workers > 0))
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=n_workers, pin_memory=pin,
                              persistent_workers=(n_workers > 0))
    return train_loader, val_loader, train_ds.classes

## Training Loop

- `train_one_epoch` shows a **per-batch tqdm progress bar** with live loss/acc.
- `train_model` saves a **checkpoint after every epoch** and resumes from it if interrupted.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch, num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    desc = f"Epoch {epoch+1:2d}/{num_epochs} [Train]"
    it   = tqdm(dataloader, desc=desc, leave=True) if HAS_TQDM else dataloader

    for batch_idx, (inputs, labels) in enumerate(it):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted  = torch.max(outputs, 1)
        correct       += (predicted == labels).sum().item()
        total         += labels.size(0)

        if HAS_TQDM:
            it.set_postfix(loss=f"{running_loss/total:.4f}",
                           acc=f"{correct/total*100:.2f}%")
        elif (batch_idx + 1) % 20 == 0:
            print(f"  [{epoch+1}/{num_epochs}] Batch {batch_idx+1}/{len(dataloader)} "
                  f"| Loss: {running_loss/total:.4f} | Acc: {correct/total*100:.2f}%")

    return running_loss / total, correct / total * 100

In [ ]:
def evaluate(model, dataloader, criterion, device, epoch, num_epochs):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    desc = f"Epoch {epoch+1:2d}/{num_epochs} [Val]  "
    it   = tqdm(dataloader, desc=desc, leave=True) if HAS_TQDM else dataloader

    with torch.no_grad():
        for inputs, labels in it:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted  = outputs.max(1)
            correct       += predicted.eq(labels).sum().item()
            total         += labels.size(0)

            if HAS_TQDM:
                it.set_postfix(loss=f"{running_loss/total:.4f}",
                               acc=f"{correct/total*100:.2f}%")

    return running_loss / total, correct / total * 100

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device,
                num_epochs, scheduler=None, model_name="Model",
                checkpoint_dir: Optional[str] = None,
                backup_checkpoint_dir: Optional[str] = None):

    history  = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    start_epoch = 0

    def _ckpt_path(d):
        return Path(d) / f"{model_name}_checkpoint.pth" if d else None

    primary_path = _ckpt_path(checkpoint_dir)
    backup_path  = _ckpt_path(backup_checkpoint_dir)

    # ── Resume: try primary first, then backup ────────────────────────────────
    resume_path = None
    for p in [primary_path, backup_path]:
        if p and p.exists():
            resume_path = p
            break

    if resume_path:
        print(f"Checkpoint found: {resume_path}")
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if scheduler and ckpt.get("scheduler_state_dict"):
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        history     = ckpt["history"]
        best_acc    = ckpt["best_acc"]
        best_wts    = ckpt["best_model_wts"]
        start_epoch = ckpt["epoch"] + 1
        print(f"Resuming {model_name} from epoch {start_epoch + 1}/{num_epochs}")
    else:
        print(f"\n{'='*60}")
        print(f"Training {model_name}")
        print(f"{'='*60}")

    if start_epoch >= num_epochs:
        print(f"{model_name} already fully trained ({num_epochs} epochs). Skipping.")
        model.load_state_dict(best_wts)
        return model, history

    start = time.time()
    for epoch in range(start_epoch, num_epochs):
        t0 = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, epoch, num_epochs)
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device, epoch, num_epochs)

        if scheduler:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())

        print(f"  >> Epoch {epoch+1:2d}/{num_epochs} summary | "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.2f}% | "
              f"Best Val: {best_acc:.2f}% | Time: {time.time()-t0:.1f}s")

        # ── Save checkpoint to primary and backup after every epoch ───────────
        ckpt_data = {
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "history":              history,
            "best_acc":             best_acc,
            "best_model_wts":       best_wts,
        }
        for path in filter(None, [primary_path, backup_path]):
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(ckpt_data, path)

    print(f"\nDone in {(time.time()-start)/60:.1f} min — best val acc: {best_acc:.2f}%")
    model.load_state_dict(best_wts)
    return model, history

## Visualisation

In [ ]:
def plot_curves(history: Dict, title: str = "Training History"):
    epochs     = list(range(1, len(history["train_loss"]) + 1))
    best_epoch = history["val_acc"].index(max(history["val_acc"])) + 1
    best_val   = max(history["val_acc"])
    BLUE, RED, GRN = "#1565C0", "#E53935", "#43A047"

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker="o", color=BLUE)
    axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   marker="s", color=RED)
    axes[0].axvline(best_epoch, color=GRN, linestyle="--", alpha=0.6)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Loss Curve")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[0].spines[["top", "right"]].set_visible(False)

    axes[1].plot(epochs, history["train_acc"], label="Train Acc", marker="o", color=BLUE)
    axes[1].plot(epochs, history["val_acc"],   label="Val Acc",   marker="s", color=RED)
    axes[1].axvline(best_epoch, color=GRN, linestyle="--", alpha=0.6,
                    label=f"Best epoch ({best_epoch})")
    axes[1].scatter([best_epoch], [best_val], color=GRN, s=120, zorder=5)
    axes[1].annotate(
        f"Best val acc\n{best_val:.1f}%",
        xy=(best_epoch, best_val), xytext=(best_epoch + 0.4, best_val - 6),
        fontsize=9, color=GRN, fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=GRN),
    )
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)"); axes[1].set_title("Accuracy Curve")
    axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0, 105)
    axes[1].spines[["top", "right"]].set_visible(False)

    summary = (
        f"Final train acc: {history['train_acc'][-1]:.1f}%   |   "
        f"Best val acc: {best_val:.1f}%  (epoch {best_epoch})   |   "
        f"Improvement from epoch 1: {best_val - history['val_acc'][0]:+.1f}%"
    )
    fig.text(0.5, -0.04, summary, ha="center", fontsize=10, fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#E8F5E9", edgecolor="#43A047"))
    plt.tight_layout()
    plt.savefig(f"{title.replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
def plot_comparison(results: Dict[str, Dict]):
    import pandas as pd
    COLORS = plt.cm.tab10.colors

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Model Comparison — Validation Metrics", fontsize=14, fontweight="bold")

    rows = []
    for i, (name, data) in enumerate(results.items()):
        h          = data["history"]
        epochs     = list(range(1, len(h["val_acc"]) + 1))
        c          = COLORS[i % len(COLORS)]
        best_val   = max(h["val_acc"])
        best_epoch = h["val_acc"].index(best_val) + 1

        axes[0].plot(epochs, h["val_loss"], label=name, color=c, marker="o")
        axes[1].plot(epochs, h["val_acc"],  label=name, color=c, marker="s")
        axes[1].scatter([best_epoch], [best_val], color=c, s=100, zorder=5)

        rows.append({
            "Model":             name,
            "Best Val Acc (%)":  f"{best_val:.2f}",
            "Best Epoch":        best_epoch,
            "Final Val Acc (%)": f"{h['val_acc'][-1]:.2f}",
            "LR":                data["config"].lr,
        })

    for ax, ylabel, title in zip(
        axes,
        ["Loss", "Accuracy (%)"],
        ["Validation Loss", "Validation Accuracy"],
    ):
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel); ax.set_title(title)
        ax.legend(); ax.grid(True, alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    df = pd.DataFrame(rows).set_index("Model")
    print()
    print("Comparison Summary")
    print("=" * 60)
    print(df.to_string())

In [ ]:
def show_predictions(model, dataloader, class_names, mean, std, device, num_images=8):
    model.eval()
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t  = torch.tensor(std).view(3, 1, 1)

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.flatten()
    shown = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, predicted = model(inputs).max(1)

            for i in range(inputs.size(0)):
                if shown >= num_images:
                    break
                img = (inputs[i].cpu() * std_t + mean_t).permute(1, 2, 0).numpy()
                img = np.clip(img, 0, 1)
                true_lbl = class_names[labels[i]]
                pred_lbl = class_names[predicted[i]]
                axes[shown].imshow(img)
                axes[shown].set_title(
                    f"True: {true_lbl}\nPred: {pred_lbl}",
                    color="green" if true_lbl == pred_lbl else "red", fontsize=9,
                )
                axes[shown].axis("off")
                shown += 1

            if shown >= num_images:
                break

    plt.tight_layout()
    plt.show()

## High-level API

`finetune_model` — trains one backbone end-to-end from a `ModelConfig`.  
`compare_models` — iterates over a list of configs, trains each, then plots the comparison.

Both accept `checkpoint_dir` — set it to `CHECKPOINT_DIR` to enable resume.

In [ ]:
def finetune_model(config: ModelConfig, train_loader, val_loader,
                   num_classes: int, device: torch.device,
                   checkpoint_dir: Optional[str] = None,
                   backup_checkpoint_dir: Optional[str] = None):
    model = build_model(config, num_classes).to(device)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{config.name}: {trainable:,} / {total:,} trainable params ({trainable/total:.1%})")

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.t_max)

    model, history = train_model(
        model, train_loader, val_loader, criterion, optimizer, device,
        config.epochs, scheduler=scheduler, model_name=config.name,
        checkpoint_dir=checkpoint_dir,
        backup_checkpoint_dir=backup_checkpoint_dir,
    )
    return model, history

In [ ]:
def compare_models(
    configs: List[ModelConfig],
    train_dir: str,
    val_dir: str,
    device: Optional[torch.device] = None,
    checkpoint_dir: Optional[str] = None,
    backup_checkpoint_dir: Optional[str] = None,
) -> Dict[str, Dict]:
    if device is None:
        device = get_device()

    results: Dict[str, Dict] = {}

    for config in configs:
        print()
        print("=" * 60)
        print(f"Starting: {config.name}  (batch_size={config.batch_size})")
        print("=" * 60)

        train_loader, val_loader, classes = get_dataloaders(
            train_dir, val_dir, config.mean, config.std,
            config.resize_size, config.batch_size, device
        )
        model, history = finetune_model(
            config, train_loader, val_loader, len(classes), device,
            checkpoint_dir=checkpoint_dir,
            backup_checkpoint_dir=backup_checkpoint_dir,
        )
        results[config.name] = {
            "model":   model,
            "history": history,
            "config":  config,
            "classes": classes,
        }
        plot_curves(history, title=f"{config.name} Training History")

    plot_comparison(results)
    return results

## Run Experiments

Define one `ModelConfig` per backbone, then call `compare_models`.  
Pass `checkpoint_dir=CHECKPOINT_DIR` to enable resume on Colab disconnects.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Ordered fast → slow so you see results sooner and can abort early if needed.
# batch_size tuned for 8 GB VRAM (full fine-tuning, no AMP, 224x224 input).
configs = [
    ModelConfig(                          # ~3.4M params — fastest
        name="mobilenet_v2",
        mean=IMAGENET_MEAN, std=IMAGENET_STD,
        lr=3e-4, weight_decay=1e-4, epochs=10,
        batch_size=128,
    ),
    ModelConfig(                          # ~5.3M params — fast
        name="efficientnet_b0",
        mean=IMAGENET_MEAN, std=IMAGENET_STD,
        lr=3e-4, weight_decay=1e-4, epochs=10,
        batch_size=128,
    ),
    ModelConfig(                          # ~11M params — moderate
        name="resnet18",
        mean=IMAGENET_MEAN, std=IMAGENET_STD,
        lr=1e-4, weight_decay=1e-4, epochs=10,
        batch_size=128,
    ),
    ModelConfig(                          # ~25M params — slowest
        name="resnet50",
        mean=IMAGENET_MEAN, std=IMAGENET_STD,
        lr=1e-4, weight_decay=1e-4, epochs=10,
        batch_size=64,
        resize_size=232,
    ),
]

In [ ]:
results = compare_models(
    configs,
    train_dir=DATA_TRAIN_DIR,
    val_dir=DATA_VAL_DIR,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    backup_checkpoint_dir=BACKUP_CHECKPOINT_DIR,
)

## Save the Best Model

After reviewing the comparison, call `save_model` to persist whichever backbone won.

In [ ]:
def save_model(results: Dict[str, Dict], model_name: str, save_dir: str = MODEL_SAVE_DIR):
    data      = results[model_name]
    save_path = Path(save_dir)
    save_path.mkdir(exist_ok=True)

    weights_path = save_path / f"freshguard_{model_name}.pth"
    classes_path = save_path / "class_names.json"

    torch.save(data["model"].state_dict(), weights_path)
    with open(classes_path, "w") as f:
        json.dump(data["classes"], f, indent=2)

    print(f"Weights : {weights_path}")
    print(f"Classes : {classes_path} ({len(data['classes'])} classes)")

# Example — pick the winner after reviewing the comparison chart:
# save_model(results, "resnet50")